## 0 · Setup prostředí

Workbench běží na RHOAI _Standard Data Science Notebook_ image. Pandas,
numpy, scikit-learn, matplotlib, boto3 a od RHOAI 3.2 i **MLflow klient**
(Red Hat build, viz [KCS 7136121](https://access.redhat.com/articles/7136121))
jsou už předinstalované. Žádný `pip install` se v notebooku nedělá.

Platforma navíc workbench podu sama nastavuje `MLFLOW_TRACKING_URI`,
`MLFLOW_TRACKING_AUTH=kubernetes-namespaced` a namountuje SA token —
tracking server je tím pádem k mání bez jediného řádku konfigurace.

In [ ]:
# Ověř, že workbench je naladěn proti platform-managed MLflow.
# RHOAI 3.4 přidá tyhle env vars automaticky, když je `mlflowoperator`
# zapnutý v DataScienceCluster (kontrola: `oc get dsc -A`).
import os, mlflow
print(f'mlflow client      : {mlflow.__version__}')
print(f'MLFLOW_TRACKING_URI: {os.environ.get("MLFLOW_TRACKING_URI", "(unset)")}')
print(f'MLFLOW_TRACKING_AUTH: {os.environ.get("MLFLOW_TRACKING_AUTH", "(unset)")}')

# 03 · Skórování nových faktur a review queue

**Scénář ze slide 20:** _most mezi daty a byznysem_ · _AI quickstart pattern_

Tento notebook už cílí přímo na business uživatele — controller dostane
**seřazený seznam faktur ke schválení / odmítnutí** s vysvětlením, proč
je model označil. Žádné F1 score, žádné konfuzní matice.

## 3.1 Načtení modelu a nových faktur

In [ ]:
import os, sys, joblib, io
sys.path.insert(0, '../src')
from invoice_features import (load_invoices, build_features, explain_row,
                              FEATURE_COLUMNS)
import pandas as pd

# --- Načti model ---------------------------------------------------
if os.environ.get('AWS_S3_BUCKET'):
    from invoice_features import get_s3_client, get_bucket
    s3 = get_s3_client()
    body = s3.get_object(Bucket=get_bucket(),
                         Key='models/invoice_anomaly_model.joblib')['Body'].read()
    artifact = joblib.load(io.BytesIO(body))
else:
    artifact = joblib.load('../data/invoice_anomaly_model.joblib')

pipe = artifact['pipeline']
print('Model loaded.  ROC-AUC =', artifact['metrics']['roc_auc'])

In [ ]:
# --- Načti nové faktury -------------------------------------------
if os.environ.get('AWS_S3_BUCKET'):
    new_df = load_invoices('invoices_new_batch.csv', s3_client=s3,
                           bucket=get_bucket())
else:
    new_df = load_invoices('../data/invoices_new_batch.csv')

print(f'Nových faktur ke skórování: {len(new_df)}')
new_df.head()

## 3.2 Skórování

Vlastní inference je jednořádek. **Featury jdou přes stejnou funkci**,
jakou jsme použili při tréninku, s předaným category_index a
vendor_freq_table z train fáze — proto je výsledek deterministický.

In [ ]:
X_new, _ = build_features(
    new_df,
    category_index=artifact['category_index'],
    vendor_freq_table=pd.Series(artifact['vendor_freq_table']),
    category_stats=pd.DataFrame.from_dict(artifact['category_stats'], orient='index'),
)

new_df['anomaly_score'] = -pipe.decision_function(X_new)
new_df['is_flagged']    = (pipe.predict(X_new) == -1).astype(int)

print(f"Flagnuto: {new_df['is_flagged'].sum()} z {len(new_df)}")

## 3.3 Review queue pro controllera

Klíčový krok pro byznys: ke každé podezřelé faktuře přidáme **lidsky
čitelný důvod**. Bez něj je model černá skříňka a uživatel ho nepoužije.

In [ ]:
# Připravíme feature dataframe ve stejném tvaru, jaký dostává explain_row
feat_df = pd.DataFrame(X_new, columns=FEATURE_COLUMNS).astype(float)
feat_df['amount_czk'] = new_df['amount_czk'].values

queue = new_df.copy()
queue['reason'] = feat_df.apply(explain_row, axis=1)

queue = (queue
    .sort_values('anomaly_score', ascending=False)
    .loc[:, ['invoice_id', 'vendor', 'amount_czk', 'issued_on',
             'po_number', 'approver', 'anomaly_score', 'reason']]
    .reset_index(drop=True))

queue.head(15)

## 3.4 LLM vysvětlovač — „proč právě tahle faktura?“

Deterministické reason codes ze sloupce `reason` jsou věcné, ale suché.
Controllerovi se snáz čte celá věta. Pro top-25 podezřelých faktur
necháme LLM napsat krátké česky znějící zdůvodnění — striktně **jen**
na základě dat z faktury a reason codes, aby si model nevymýšlel.

**Endpoint:** `phi-4-quantizedw8a8-version-1` (vLLM, OpenAI-kompatibilní API)
běží jako `InferenceService` v namespace `model-test`. Workbench na něj
sahá in-cluster přes Service DNS. Pokud chceš jiný model nebo endpoint,
stačí přepsat env vars `LLM_ENDPOINT` / `LLM_MODEL` v `deploy/04-workbench.yaml`
(viz `src/explain_with_llm.py`).

Latence: cca 3-5 s na řádek na A10G — 25 řádků = ~1-2 min. Pokud je
endpoint nedostupný, funkce vrátí fallback hlášku a notebook pokračuje.

In [ ]:
from explain_with_llm import LLMClient, explain_queue

client = LLMClient()  # čte LLM_ENDPOINT / LLM_MODEL / LLM_API_KEY z env
print(f'LLM endpoint: {client.endpoint}')
print(f'LLM model   : {client.model}')

# Sanity-check: zjisti, jestli endpoint vůbec odpovídá.
try:
    served = [m.get('id') for m in client.list_models().get('data', [])]
    print('Dostupné modely:', served)
except Exception as e:
    print('Endpoint neodpovídá — vysvětlení se přeskočí. Důvod:', e)
    served = []

In [ ]:
# Vygeneruj vysvětlení pro top-25 podezřelých faktur.
# explain_queue() je defenzivní — pokud LLM padne, vrátí fallback string.
if served:
    explanations = explain_queue(queue, top_n=25, client=client)
    queue['vysvětlení'] = ''
    queue.loc[:len(explanations) - 1, 'vysvětlení'] = explanations
else:
    queue['vysvětlení'] = '(LLM endpoint nedostupný — viz README)'

queue.head(5)[['vendor', 'amount_czk', 'anomaly_score',
               'reason', 'vysvětlení']]

## 3.5 Export pro controllera

Výsledek pošleme do bucketu (a do CSV pro Excel). Controller dostane
ráno e-mailem soubor s top 25 položkami ke kontrole — včetně sloupce
`vysvětlení` z LLM.

In [ ]:
import pandas as pd
from datetime import datetime

stamp = datetime.now().strftime('%Y-%m-%d')
out_file = f'../data/review_queue_{stamp}.csv'
queue.head(25).to_csv(out_file, index=False)
print('Review queue uložena do:', out_file)

if os.environ.get('AWS_S3_BUCKET'):
    from invoice_features import write_csv_s3
    write_csv_s3(queue.head(25), s3, get_bucket(),
                 f'review_queue/{stamp}.csv')
    print('A taky do S3:', f'review_queue/{stamp}.csv')

## 3.6 Registrace modelu do Model Registry

Pokud controller potvrdí, že top položky dávají smysl, model přejde do
produkce. To vyřeší Red Hat AI **Model Registry** — promo `Staging` →
`Production` je jeden klik (nebo jeden API call).

Workbench dostává `MODEL_REGISTRY_URL` z `deploy/04-workbench.yaml`
(externí route registru, protože in-cluster Service má NetworkPolicy
omezenou na router). Auth: bearer SA token + ověření TLS přes
workbench CA bundle. RBAC zařizuje `deploy/07-model-registry-rbac.yaml`
(navázání SA na Role `registry-user-elos-model-registry`).

In [ ]:
# --- Registrace modelu do Red Hat AI Model Registry ----------------
# Idempotentní: hledáme po jméně (registry v1alpha3 ignoruje ?name=,
# musíme filtrovat client-side), a teprve když chybí, založíme.
import os, requests

registry_url = os.environ.get('MODEL_REGISTRY_URL', '').rstrip('/')
TOKEN_PATH = '/var/run/secrets/kubernetes.io/serviceaccount/token'
CA_BUNDLE  = '/etc/pki/tls/custom-certs/ca-bundle.crt'
API_BASE   = '/api/model_registry/v1alpha3'
MODEL_NAME = 'invoice-anomaly-detector'

def _find_by_name(api, name, headers, verify):
    """List + client-side filter — v1alpha3 ?name= is ignored."""
    page = requests.get(f'{api}/registered_models',
                        headers=headers, verify=verify, timeout=10)
    page.raise_for_status()
    for m in page.json().get('items', []):
        if m.get('name') == name:
            return m['id']
    return None

if registry_url and os.path.exists(TOKEN_PATH):
    with open(TOKEN_PATH) as f:
        token = f.read().strip()
    headers = {'Authorization': f'Bearer {token}',
               'Content-Type': 'application/json'}
    verify  = CA_BUNDLE if os.path.exists(CA_BUNDLE) else True
    api     = f'{registry_url}{API_BASE}'

    try:
        rm_id = _find_by_name(api, MODEL_NAME, headers, verify)
        if rm_id is None:
            r = requests.post(f'{api}/registered_models',
                              headers=headers, verify=verify, timeout=10,
                              json={
                                  'name': MODEL_NAME,
                                  'description': 'Isolation Forest pro detekci '
                                                 'anomálií ve fakturách.',
                                  'owner': 'controlling@example.com',
                              })
            r.raise_for_status()
            rm_id = r.json()['id']
            print(f'Registered model vytvořen: {MODEL_NAME}  (id={rm_id})')
        else:
            print(f'Registered model už existuje: {MODEL_NAME}  (id={rm_id})')

        # Link this training run as a ModelVersion under that model.
        if artifact.get('mlflow_run_id'):
            run_id  = artifact['mlflow_run_id']
            roc_auc = artifact['metrics']['roc_auc']
            mv = {
                'name': f'iforest-{run_id[:8]}',
                'description': f'ROC-AUC {roc_auc:.3f}',
                'state': 'LIVE',
                'author': 'controlling@example.com',
                'registeredModelId': rm_id,  # NB required v body i v cestě
                'customProperties': {
                    'mlflow_run_id': {
                        'string_value': run_id,
                        'metadataType': 'MetadataStringValue',
                    },
                },
            }
            v = requests.post(f'{api}/registered_models/{rm_id}/versions',
                              headers=headers, json=mv,
                              verify=verify, timeout=10)
            if v.status_code == 201:
                print(f'Model version vytvořena: {mv["name"]} '
                      f'(id={v.json()["id"]})')
            elif v.status_code == 409:
                print(f'Model version už existuje: {mv["name"]}')
            else:
                print(f'Version POST -> {v.status_code}: {v.text[:200]}')
        else:
            print('mlflow_run_id chybí v artifactu — verze neregistrována.')
    except Exception as e:
        print('Registry call failed (OK pro lokální spuštění):', e)
else:
    print('MODEL_REGISTRY_URL nenastaveno nebo SA token chybí — '
          'přeskakuji (běh mimo cluster).')

## 3.7 Shrnutí — co tím controller dostal

1. Ráno mu přistál v inboxu CSV s top 25 podezřelými fakturami.
2. U každé položky vidí **proč** byla označena — reason codes
   (kulatá částka, nový dodavatel s vysokou částkou, …) a nahoře
   k tomu krátké česky znějící vysvětlení od LLM.
3. Pokud potvrdí, že model funguje, BI tým ho promotne do Production.
4. Když přijde NIS2 auditor, evidence je kompletní:
   `git log` (kód) → MLflow run (parametry + metriky) →
   Model Registry (kdo a kdy schválil) → review_queue/*.csv (výstupy).

Tohle je celý byznys příběh slide 20, zhmotněný v jednom workbenchi.